# MediaScope IA — Colab Edition

**Authors:** Ricardo Torres López and Cesar Eduardo Inda Ceniceros  
**Project:** Multimodal analysis and narration of image and audio

This notebook runs the complete MediaScope project in Google Colab. It uses:

- **YOLO11n** for object detection.
- **BLIP** for image captioning.
- **Whisper** for speech recognition.
- **AST** for environmental sound classification.
- **MMS-TTS** for Spanish narration.
- **Gradio** for the interactive interface.

Run the cells from top to bottom. A **T4 GPU** is recommended. The first run is slower because the models are downloaded.

## 1. Check the Colab runtime

Before running this cell, select **Runtime > Change runtime type > T4 GPU**. The project can use CPU, but it will be much slower.

In [1]:
import platform
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)

if DEVICE == "cuda":
    gpu_name = torch.cuda.get_device_name(0)
    gpu_vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print("GPU:", gpu_name)
    print(f"Total VRAM: {gpu_vram:.2f} GB")
else:
    print("Warning: GPU was not detected.")
    print("Use Runtime > Change runtime type > T4 GPU.")

Python: 3.13.15
PyTorch: 2.11.0+cu128
Device: cuda
GPU: Tesla T4
Total VRAM: 14.56 GB


## 2. Install the dependencies

Colab already includes PyTorch. This cell does **not** reinstall it, which helps avoid CUDA conflicts. Restart the runtime only if Colab explicitly asks for it.

In [2]:
%pip install -q \
    "gradio>=5.49,<7" \
    "Pillow>=10,<13" \
    "ultralytics>=8.4,<9" \
    "transformers>=4.49,<5" \
    "accelerate>=1,<2" \
    "librosa>=0.11,<1" \
    "soundfile>=0.13,<1" \
    "scipy>=1.13,<2" \
    sentencepiece safetensors

print("Dependencies installed.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## 3. Mount Google Drive

The notebook first looks for the project ZIP in:

`MyDrive/TAE_IA_M6/Multimodal_system_main.zip`

If the file is not there, the next cell opens a manual upload window.

In [3]:
from google.colab import drive

drive.mount("/content/drive")
print("Google Drive mounted.")

Mounted at /content/drive
Google Drive mounted.


## 4. Locate or upload the project ZIP

You can change `DRIVE_ZIP` if the ZIP is stored in a different Drive folder.

In [4]:
from pathlib import Path
from google.colab import files

ZIP_NAME = "Multimodal_system_main.zip"
DRIVE_ZIP = Path("/content/drive/MyDrive/TAE_IA_M6") / ZIP_NAME

if DRIVE_ZIP.exists():
    PROJECT_ZIP = DRIVE_ZIP
    print("Project found in Drive:", PROJECT_ZIP)
else:
    print("The ZIP was not found in the expected Drive folder.")
    print("Select the MediaScope ZIP now.")
    uploaded = files.upload()
    zip_files = [Path("/content") / name for name in uploaded if name.lower().endswith(".zip")]
    if not zip_files:
        raise FileNotFoundError("No ZIP file was uploaded.")
    PROJECT_ZIP = zip_files[0]
    print("Uploaded project:", PROJECT_ZIP)

Project found in Drive: /content/drive/MyDrive/TAE_IA_M6/Multimodal_system_main.zip


## 5. Extract and configure MediaScope

This cell extracts a clean copy to the Colab temporary disk and automatically finds `MediaScope_Etapa7`. Files stored in `/content` disappear when the Colab session ends; the original ZIP in Drive is not modified.

In [5]:
import os
import shutil
import sys
import zipfile
from pathlib import Path

EXTRACT_ROOT = Path("/content/mediascope_project")
if EXTRACT_ROOT.exists():
    shutil.rmtree(EXTRACT_ROOT)
EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(PROJECT_ZIP, "r") as archive:
    archive.extractall(EXTRACT_ROOT)

matches = list(EXTRACT_ROOT.rglob("MediaScope_Etapa7"))
if not matches:
    raise FileNotFoundError("MediaScope_Etapa7 was not found inside the ZIP.")

PROJECT_ROOT = matches[0]
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

required_files = [
    "app.py",
    "config.py",
    "services/vision.py",
    "services/audio.py",
    "services/fusion.py",
    "services/tts.py",
    "services/exporter.py",
    "utils/validation.py",
]
missing = [name for name in required_files if not (PROJECT_ROOT / name).exists()]
if missing:
    raise FileNotFoundError("Missing project files: " + ", ".join(missing))

print("Project root:", PROJECT_ROOT)
print("Output folder:", OUTPUT_DIR)
print("All required modules were found.")

Project root: /content/mediascope_project/Multimodal-system-for-integrated-image-and-audio-analysis-and-narration_TAE_IA-main/MediaScope_Etapa7
Output folder: /content/mediascope_project/Multimodal-system-for-integrated-image-and-audio-analysis-and-narration_TAE_IA-main/MediaScope_Etapa7/outputs
All required modules were found.


## 6. Import and verify the pipeline

This is a light test. It verifies the modules without loading all model weights. The models use lazy loading when an analysis starts.

In [6]:
import importlib
import gradio as gr
import transformers
import ultralytics

import config
import app
from services.vision import analyze_image
from services.audio import analyze_audio
from services.fusion import build_multimodal_report
from services.tts import synthesize_report
from services.exporter import export_markdown_report

print("MediaScope:", config.APP_TITLE)
print("Gradio:", gr.__version__)
print("Transformers:", transformers.__version__)
print("Ultralytics:", ultralytics.__version__)
print("Detection model:", config.VISION_DETECTION_MODEL)
print("Caption model:", config.VISION_CAPTION_MODEL)
print("ASR model:", config.SPEECH_RECOGNITION_MODEL)
print("Sound model:", config.SOUND_CLASSIFICATION_MODEL)
print("TTS model:", config.TEXT_TO_SPEECH_MODEL)
print("Pipeline modules imported correctly.")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
MediaScope: MediaScope
Gradio: 6.17.3
Transformers: 4.57.6
Ultralytics: 8.4.163
Detection model: yolo11n.pt
Caption model: Salesforce/blip-image-captioning-base
ASR model: openai/whisper-base
Sound model: MIT/ast-finetuned-audioset-10-10-0.4593
TTS model: facebook/mms-tts-spa
Pipeline modules imported correctly.


## 7. Optional YOLO comparison

This experiment compares **YOLO11n, YOLO11s, and YOLO11m** under the same conditions. It helps justify which detector gives the best balance between detections, time, and GPU memory.

Set `RUN_YOLO_COMPARISON = True`. The cell will ask for one test image. For a stronger report, repeat the experiment with several images and save the tables.

In [7]:
RUN_YOLO_COMPARISON = True

if RUN_YOLO_COMPARISON:
    import gc
    import time
    import pandas as pd
    from PIL import Image
    from google.colab import files
    from ultralytics import YOLO

    uploaded = files.upload()
    image_names = [name for name in uploaded if name.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))]
    if not image_names:
        raise FileNotFoundError("Upload one JPG, PNG, or WEBP image.")

    test_image = Image.open(image_names[0]).convert("RGB")
    rows = []

    for model_name in ["yolo11n.pt", "yolo11s.pt", "yolo11m.pt"]:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()

        model = YOLO(model_name)
        model.predict(test_image, imgsz=640, conf=0.35, device=0 if torch.cuda.is_available() else "cpu", verbose=False)

        times_ms = []
        detections = []
        mean_confidences = []
        for _ in range(5):
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            start = time.perf_counter()
            result = model.predict(
                test_image,
                imgsz=640,
                conf=0.35,
                device=0 if torch.cuda.is_available() else "cpu",
                verbose=False,
            )[0]
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            times_ms.append((time.perf_counter() - start) * 1000)
            count = 0 if result.boxes is None else len(result.boxes)
            detections.append(count)
            if count:
                mean_confidences.append(float(result.boxes.conf.mean().item()))

        peak_vram_mb = (
            torch.cuda.max_memory_allocated() / 1024**2
            if torch.cuda.is_available()
            else None
        )
        rows.append({
            "model": model_name,
            "detections": round(sum(detections) / len(detections), 1),
            "mean_confidence": round(sum(mean_confidences) / len(mean_confidences), 3) if mean_confidences else 0.0,
            "mean_time_ms": round(sum(times_ms) / len(times_ms), 1),
            "min_time_ms": round(min(times_ms), 1),
            "max_time_ms": round(max(times_ms), 1),
            "peak_vram_mb": round(peak_vram_mb, 1) if peak_vram_mb is not None else "CPU",
        })

        del model, result
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    yolo_comparison = pd.DataFrame(rows)
    display(yolo_comparison)
    yolo_comparison.to_csv(OUTPUT_DIR / "yolo_comparison.csv", index=False)
    print("Saved:", OUTPUT_DIR / "yolo_comparison.csv")
else:
    print("YOLO comparison skipped. Set RUN_YOLO_COMPARISON = True to run it.")

Saving 02_car.jpg to 02_car.jpg
Saving 03_dog.jpg to 03_dog.jpg
Saving 07_cats&dogs.png to 07_cats&dogs.png
Saving 07_fishing.jpg to 07_fishing.jpg
Saving 08_street.png to 08_street.png
Saving 09_classroom.png to 09_classroom.png


,model,detections,mean_confidence,mean_time_ms,min_time_ms,max_time_ms,peak_vram_mb
0,yolo11n.pt,1.0,0.771,13.5,10.3,19.3,56.4
1,yolo11s.pt,1.0,0.838,14.0,13.8,14.4,94.0
2,yolo11m.pt,2.0,0.666,27.3,27.1,27.8,160.2


Saved: /content/mediascope_project/Multimodal-system-for-integrated-image-and-audio-analysis-and-narration_TAE_IA-main/MediaScope_Etapa7/outputs/yolo_comparison.csv


## 8. Launch the MediaScope interface

The first analysis downloads and loads the selected models, so it may take several minutes. Keep this cell running while you use the public Gradio link.

Recommended test order:

1. Image only.
2. Audio only.
3. Image and audio together.
4. No input, to confirm the validation message.

Use files shorter than 30 seconds and start with the default confidence of 0.35.

In [8]:
from app import CUSTOM_CSS, build_app, build_theme, gradio_major_version

demo = build_app()
demo.queue(default_concurrency_limit=1, max_size=20)

launch_options = {
    "share": True,
    "debug": True,
    "show_error": True,
}

if gradio_major_version() >= 6:
    launch_options.update(theme=build_theme(), css=CUSTOM_CSS)

demo.launch(**launch_options)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://44b5088a47f564facc.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/47.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/145M [00:00<?, ?B/s]

Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/gradio/queueing.py", line 867, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/gradio/route_utils.py", line 386, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<11 lines>...
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/gradio/blocks.py", line 2277, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<8 lines>...
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/gradio/blocks.py", line 1654, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        fn, *processed_input, limiter=self.limiter
        ^^^^^^^^^^^^^^

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://44b5088a47f564facc.gradio.live


## 9. Save the outputs to Google Drive

Run this cell after the tests. It copies generated narrations, Markdown reports, and the optional YOLO comparison to Drive.

In [9]:
import shutil
from datetime import datetime
from pathlib import Path

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
DRIVE_OUTPUT = Path("/content/drive/MyDrive/TAE_IA_M6/MediaScope_outputs") / timestamp
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)

copied = 0
for source in OUTPUT_DIR.glob("*"):
    if source.is_file():
        shutil.copy2(source, DRIVE_OUTPUT / source.name)
        copied += 1

print(f"Copied {copied} output file(s).")
print("Drive folder:", DRIVE_OUTPUT)

Copied 6 output file(s).
Drive folder: /content/drive/MyDrive/TAE_IA_M6/MediaScope_outputs/20260926_170604


## 10. Notes for the report

- Record the GPU name, confidence threshold, input duration, and package versions.
- Run one warm-up inference before measuring time.
- Compare the same files under the same conditions.
- Report model errors and disagreements between YOLO and BLIP.
- Do not treat automatic predictions as verified facts.
- Close the public Gradio link when the demonstration is complete.